# CS2311 Special Topic in Computer Science — Agent0 cho sinh code: benchmark nhiều LLM trên Kaggle

Notebook này chạy benchmark **MBPP+ (378 bài, EvalPlus)** cho nhiều model, mỗi model với 4 cấu hình:

| Cấu hình | Ý nghĩa |
|---|---|
| `baseline` | không có ví dụ |
| `retrieved` | n ví dụ giống nhất trong knowledge base (KB) — cách của dự án |
| `random` | n ví dụ ngẫu nhiên từ KB |
| `fixed` | cùng n ví dụ cho mọi bài |

`retrieved` so với `random` / `fixed` tách tác dụng của **truy hồi** khỏi tác dụng của việc **có ví dụ**. Kết quả cuối: bảng pass@1, so sánh cặp McNemar (hiệu chỉnh Holm), biểu đồ "KB giúp model mạnh hay model yếu nhiều hơn" và bảng Δ theo độ giống của ví dụ truy hồi (kiểm tra độ liên quan / rò rỉ).

**Thí nghiệm Agent0 (tuỳ chọn, nên chạy ở phiên riêng, `AGENT0_ABLATION = True`)**: sinh KB mới theo kiểu Agent0 — p̂ đo trên *model + KB hiện có* (đồng tiến hoá không cập nhật trọng số) — rồi tách thành KB `phat` (đề có p̂ ∈ [0.25, 0.75]) và KB `ctrl` cùng cỡ chọn ngẫu nhiên. Chạy `retrieved` với hai KB: chênh lệch là tác dụng của việc chọn đề ở biên năng lực.

**Trước khi chạy** (bảng bên phải của notebook):
1. *Add Input* → dataset chứa project (zip `CS2311_kaggle_upload.zip` đã tải lên).
2. *Session options* → Accelerator **GPU T4 x2** (model local cần GPU; chỉ chạy model API thì để None cho đỡ tốn quota GPU), **Internet: On**.
3. *Add-ons → Secrets*: thêm và tick các key cần dùng: `GROQ_API_KEY`, `OLLAMA_API_KEY`, `OPENAI_API_KEY`, `GEMINI_API_KEY`, `CUSTOM_API_KEY` (không có key nào thì chỉ chạy model local).

**Chạy thử** với `LIMIT = 10` (vài phút, không chấm) rồi đặt `LIMIT = None` và bấm **Save Version → Save & Run All (Commit)** để chạy nền tối đa 12 giờ. Hết giờ thì phiên sau thêm output của lần trước làm Input, đặt `PREVIOUS_RESULTS = "auto"` rồi chạy lại — các bài đã sinh được giữ, chỉ chạy phần còn thiếu.

Với **GPU T4 x2**, model local chia hai làn: hai model chạy cùng lúc, mỗi model một GPU (`LOCAL_LANES`).

In [ ]:
# ===================== CẤU HÌNH — sửa ở đây =====================
CONFIGS = ["baseline", "retrieved", "random", "fixed"]  # bỏ bớt nếu thiếu thời gian, vd ["baseline", "retrieved"]
N_EXAMPLES = 2                     # số ví dụ few-shot
KB_NAME = "all_models_clean"       # KB truy hồi: Data/knowledge_base/<KB_NAME>/knowledge_base.jsonl (431 mục đã kiểm lại)
EXCLUDE_TEST_REPAIRED = True       # bỏ mục KB có test từng bị sửa cho khớp lời giải (all_models_clean: 431 -> 391 mục)
ONLY = None                        # vd ["gemma3-4b", "gpt-oss-120b"]: chỉ chạy các label này trong kaggle/roster.json
SKIP_LOCAL = None                  # None = tự động bỏ model local khi phiên không có GPU
TIME_BUDGET_HOURS = 10.5           # dừng sinh trước giới hạn 12 giờ/phiên của Kaggle (còn thời gian chấm + lưu)
LIMIT = None                       # vd 10 để chạy thử nhanh (không chấm, kết quả không dùng được)
PREVIOUS_RESULTS = None            # chạy tiếp: "auto" = mọi thư mục results/ của output cũ đã Add Input; hoặc đường dẫn / glob
LOCAL_LANES = None                 # số model local chạy cùng lúc; None = số GPU (T4 x2 -> 2, mỗi model một GPU)
RUN_SMOKE_TEST = False             # True: chạy tests/smoke_test.py (~2 phút) để kiểm tra môi trường
RUN_MAIN_MATRIX = True             # ma trận chính: mọi model × CONFIGS với KB_NAME
AGENT0_ABLATION = False            # True: thí nghiệm Agent0 (xem đầu notebook); nên đặt RUN_MAIN_MATRIX = False ở phiên đó
KB_BUILDER = {"provider": "ollama", "model": "qwen2.5-coder:7b",  # model sinh đề + đo p_hat (~1–3 giờ cho 300 đề trên T4)
              "synthetic_count": 300, "uncertainty_samples": 4, "workers": 4,
              "p_hat_with_kb": True,       # True: p_hat đo trên model + KB_NAME; False: p_hat của model trần như Agent0 vòng 1
              "min_kb": 20}                # cần ít nhất chừng này đề ở biên (mỗi KB phat/ctrl) mới chạy thí nghiệm
ABLATION_MODELS = ["qwen2.5-coder-7b", "qwen2.5-coder-1.5b", "gemma3-4b"]  # label trong roster chạy retrieved với KB phat/ctrl
RESULTS_DIR = "/kaggle/working/results"

In [ ]:
import os, sys, json, time, shutil, zipfile, subprocess, pathlib, urllib.request

DEADLINE = time.time() + TIME_BUDGET_HOURS * 3600   # mọi bước dài (sinh KB, các lần run_matrix) dùng chung ngân sách

def hours_left():
    return round(max(0.05, (DEADLINE - time.time()) / 3600), 2)

def run_live(cmd, cwd=None, env=None, check=False):
    """Chạy lệnh và in output ngay (subprocess thường không hiện output trong notebook)."""
    print("$", " ".join(map(str, cmd)), flush=True)
    proc = subprocess.Popen(list(map(str, cmd)), cwd=cwd, env=env, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    code = proc.wait()
    if code != 0:
        print(f"⚠ lệnh kết thúc với mã {code}", flush=True)
        if check:
            raise RuntimeError(f"lệnh lỗi (mã {code}): {cmd[:4]}")

def find_project(root=pathlib.Path("/kaggle/input")):
    for p in root.rglob("agent0/__init__.py"):
        if p.parent.parent.name == "Source_Code":
            return p.parent.parent.parent
    for z in root.rglob("*.zip"):
        with zipfile.ZipFile(z) as zf:
            if any(n.endswith("Source_Code/agent0/__init__.py") for n in zf.namelist()):
                dest = pathlib.Path("/tmp/project_zip")
                shutil.rmtree(dest, ignore_errors=True)
                zf.extractall(dest)
                return next(dest.rglob("Source_Code/agent0/__init__.py")).parent.parent.parent
    raise FileNotFoundError("Không thấy project trong /kaggle/input. Add Input -> dataset chứa CS2311_kaggle_upload.zip")

def show_inputs(root=pathlib.Path("/kaggle/input"), depth=3, max_lines=60):
    """In các thư mục trong /kaggle/input (Kaggle gắn dataset ở /kaggle/input/datasets/<user>/<slug>/...)."""
    lines = [str(p) + "/" for p in sorted(root.rglob("*")) if p.is_dir() and len(p.relative_to(root).parts) <= depth
             and "Source_Code" not in p.parts]
    print("Input:", *(lines[:max_lines] or ["(trống)"]), sep="\n  ")

def previous_results(spec, exclude):
    """Các thư mục results/ của output cũ: "auto" = mọi thư mục có <model>/<lần chạy>_samples.jsonl."""
    import glob
    if not spec:
        return []
    if spec == "auto":
        found = {p.parent.parent for p in pathlib.Path("/kaggle/input").glob("**/*/*_samples.jsonl")
                 if p.parent.parent.name == "results"}
    elif any(c in spec for c in "*?["):
        found = {pathlib.Path(p) for p in glob.glob(spec, recursive=True)}
    else:
        found = {pathlib.Path(spec)}
    found = sorted(p for p in found if exclude not in p.parents)
    missing = [p for p in found if not p.exists()]
    if missing or not found:
        raise FileNotFoundError(f"PREVIOUS_RESULTS: không thấy {missing or spec}")
    return found

show_inputs()
SRC = find_project()
PROJECT = pathlib.Path("/tmp/cs2311")          # bản chép để chạy (ghi được; /kaggle/input chỉ đọc)
shutil.rmtree(PROJECT, ignore_errors=True)
shutil.copytree(SRC, PROJECT, ignore=shutil.ignore_patterns("__pycache__", "*.pyc", ".env", "Report", "Slide"))
SOURCE = PROJECT / "Source_Code"
os.chdir(SOURCE)
pathlib.Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)
print("project:", SRC, "->", PROJECT)
print("python:", sys.version.split()[0])

In [ ]:
# Thư viện: chỉ cần EvalPlus (chấm MBPP+). Các thư viện khác đã có sẵn trên Kaggle.
run_live([sys.executable, "-m", "pip", "install", "-q", "evalplus==0.3.1", "python-dotenv"], check=True)

In [ ]:
# API key từ Kaggle Secrets -> biến môi trường (các lệnh con tự đọc). Key không bao giờ được in ra.
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
except Exception:
    _secrets = None
for name in ["GROQ_API_KEY", "OLLAMA_API_KEY", "OPENAI_API_KEY", "GEMINI_API_KEY", "CUSTOM_API_KEY"]:
    value = None
    if _secrets is not None:
        try:
            value = _secrets.get_secret(name)
        except Exception:
            value = None
    if value:
        os.environ[name] = value
    print(("✓ " if os.environ.get(name) else "· ") + name + ("" if os.environ.get(name) else " (không có)"))

In [ ]:
# GPU + Ollama local (chạy model nhỏ trên GPU, và model embedding mxbai-embed-large cho cấu hình retrieved).
HAS_GPU = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
N_GPU = 0
if HAS_GPU:
    gpus = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                          capture_output=True, text=True).stdout.strip().splitlines()
    N_GPU = len(gpus)
    print(*gpus, sep="\n")
else:
    print("Không có GPU: chỉ chạy model qua API (model local bị bỏ qua).")
if SKIP_LOCAL is None:
    SKIP_LOCAL = not HAS_GPU
if LOCAL_LANES is None:
    LOCAL_LANES = max(1, N_GPU)
print("Model local chạy cùng lúc:", LOCAL_LANES)

if not shutil.which("ollama"):
    run_live(["bash", "-c", "apt-get -qq update >/dev/null && apt-get -qq install -y zstd pciutils lshw >/dev/null; "
                            "curl -fsSL https://ollama.com/install.sh | sh"])
# Mỗi làn một model sinh + model embedding; KEEP_ALIVE ngắn để model của cấu hình trước nhường bộ nhớ GPU
ollama_env = dict(os.environ, OLLAMA_NUM_PARALLEL="4", OLLAMA_MAX_LOADED_MODELS=str(LOCAL_LANES + 1),
                  OLLAMA_CONTEXT_LENGTH="8192", OLLAMA_KEEP_ALIVE="10m")
ollama_env.pop("OLLAMA_API_KEY", None)   # server local không cần key cloud
OLLAMA_LOG = open(pathlib.Path(RESULTS_DIR) / "ollama_server.log", "a")
OLLAMA = subprocess.Popen(["ollama", "serve"], stdout=OLLAMA_LOG, stderr=subprocess.STDOUT, env=ollama_env)
for _ in range(90):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama không khởi động được — xem results/ollama_server.log")
run_live(["ollama", "pull", "mxbai-embed-large"])

In [ ]:
if RUN_SMOKE_TEST:
    run_live([sys.executable, "tests/smoke_test.py"], env=dict(os.environ, PYTHONDONTWRITEBYTECODE="1"))

In [ ]:
# Danh sách model sẽ chạy + model hiện có trên Groq / Ollama Cloud (để sửa tên trong kaggle/roster.json nếu cần).
sys.path.insert(0, str(SOURCE))
from tools.run_matrix import load_roster, is_local

roster = load_roster("kaggle/roster.json", ONLY)
need = {"groq": "GROQ_API_KEY", "ollama-cloud": "OLLAMA_API_KEY", "openai": "OPENAI_API_KEY"}
print(f"{'label':28s} {'provider':13s} {'model':24s} {'nhóm':6s} sẽ chạy?")
for m in roster:
    key = m.get("api_key_env") or need.get(m["provider"])
    if is_local(m) and SKIP_LOCAL:
        verdict = "không (phiên không có GPU)"
    elif key and not os.environ.get(key):
        verdict = f"không (thiếu {key})"
    else:
        verdict = "có"
    print(f"{m['label']:28s} {m['provider']:13s} {m['model']:24s} {str(m.get('tier')):6s} {verdict}")

def _get(url, key=None):
    req = urllib.request.Request(url, headers={"Authorization": f"Bearer {key}"} if key else {})
    with urllib.request.urlopen(req, timeout=20) as r:
        return json.loads(r.read().decode())
if os.environ.get("GROQ_API_KEY"):
    try:
        ids = sorted(m["id"] for m in _get("https://api.groq.com/openai/v1/models", os.environ["GROQ_API_KEY"])["data"])
        print("\nGroq hiện có:", ", ".join(ids))
    except Exception as e:
        print("Không lấy được danh sách model Groq:", e)
if os.environ.get("OLLAMA_API_KEY"):
    try:
        names = sorted(m["name"] for m in _get("https://ollama.com/api/tags", os.environ["OLLAMA_API_KEY"])["models"])
        print("\nOllama Cloud hiện có:", ", ".join(names))
    except Exception as e:
        print("Không lấy được danh sách model Ollama Cloud:", e)

In [ ]:
# Chạy tiếp từ phiên trước (nếu có): chép kết quả cũ vào RESULTS_DIR, run_matrix chỉ sinh phần còn thiếu.
for prev in previous_results(PREVIOUS_RESULTS, exclude=SRC):
    shutil.copytree(prev, RESULTS_DIR, dirs_exist_ok=True, ignore=shutil.ignore_patterns("summary", "logs*"))
    print("đã chép kết quả cũ từ", prev)
# Cache lời gọi LLM của thí nghiệm Agent0: để ở /tmp (một file mỗi lời gọi -> quá nhiều file cho output của Kaggle),
# cuối notebook nén thành llm_cache.zip; phiên sau tự giải nén lại nếu output cũ có file này.
LLM_CACHE = pathlib.Path("/tmp/llm_cache")
for z in sorted(pathlib.Path("/kaggle/input").glob("**/llm_cache.zip")):
    shutil.unpack_archive(z, LLM_CACHE)
    print("đã nạp cache LLM từ", z)

KB_PATH = PROJECT / "Data" / "knowledge_base" / KB_NAME / "knowledge_base.jsonl"
print("KB:", KB_PATH, sum(1 for _ in open(KB_PATH, encoding="utf-8")), "mục",
      "(bỏ mục có test bị sửa khi truy hồi)" if EXCLUDE_TEST_REPAIRED else "")

# (Tuỳ chọn) Thí nghiệm Agent0: sinh KB mới, p_hat đo trên model + KB_NAME, tách KB phat / ctrl cùng cỡ.
ABLATION_KBS = {}
if AGENT0_ABLATION:
    kb_new = pathlib.Path("/kaggle/working/kb_agent0")
    if KB_BUILDER["provider"] == "ollama":
        run_live(["ollama", "pull", KB_BUILDER["model"]])
    cmd = [sys.executable, "-m", "agent0.scripts.run_curriculum", "--provider", KB_BUILDER["provider"],
           "--model", KB_BUILDER["model"], "--synthetic_count", KB_BUILDER["synthetic_count"],
           "--items_per_strategy", KB_BUILDER["synthetic_count"],   # không thì chỉ ~10 đề/chiến lược được giải
           "--uncertainty_samples", KB_BUILDER["uncertainty_samples"], "--p_hat_action", "tag",
           "--skip_reasoning", "--workers", KB_BUILDER["workers"], "--output_dir", kb_new, "--no_global_kb",
           "--cache_dir", LLM_CACHE]
    if KB_BUILDER["p_hat_with_kb"]:
        cmd += ["--p_hat_kb", KB_PATH, "--p_hat_n_examples", N_EXAMPLES]
    run_live(cmd)
    built = sorted(kb_new.glob("*/knowledge_base.jsonl"))
    if built:
        split_dir = pathlib.Path("/kaggle/working/kb_split")
        run_live([sys.executable, "tools/split_kb_by_phat.py", built[0], "--out_dir", split_dir])
        report = json.loads((split_dir / "split_report.json").read_text()) if (split_dir / "split_report.json").exists() else {}
        if report.get("in_band", 0) >= KB_BUILDER.get("min_kb", 20):
            ABLATION_KBS = {"ctrl": split_dir / "ctrl" / "knowledge_base.jsonl", "phat": split_dir / "phat" / "knowledge_base.jsonl"}
        print("Tách KB:", {k: report.get(k) for k in ("tagged", "in_band", "control", "overlap")},
              "-> chạy thí nghiệm" if ABLATION_KBS else "-> quá ít đề ở biên, tăng synthetic_count rồi chạy lại")
    else:
        print("Không sinh được KB mới — xem log ở trên.")

In [ ]:
# Chạy ma trận model × cấu hình. Model local (GPU) và model API chạy song song; chấm EvalPlus chạy nền.
def matrix_cmd(configs, kb_path, kb_tag=None, only=ONLY):
    cmd = [sys.executable, "tools/run_matrix.py", "--roster", "kaggle/roster.json", "--results_dir", RESULTS_DIR,
           "--configs", *configs, "--n_examples", N_EXAMPLES, "--kb_path", kb_path, "--time_budget_hours", hours_left()]
    if EXCLUDE_TEST_REPAIRED:
        cmd += ["--exclude_test_repaired"]
    if kb_tag:
        cmd += ["--kb_tag", kb_tag]
    if only:
        cmd += ["--only", *only]
    if SKIP_LOCAL:
        cmd += ["--skip_local"]
    else:
        cmd += ["--local_lanes", LOCAL_LANES]
    if LIMIT:
        cmd += ["--limit", LIMIT]
    return cmd

if RUN_MAIN_MATRIX:
    run_live(matrix_cmd(CONFIGS, KB_PATH))
# Thí nghiệm Agent0: baseline (dùng chung, bỏ qua nếu đã có) + retrieved với KB ctrl rồi KB phat
for tag, path in ABLATION_KBS.items():
    run_live(matrix_cmd(["baseline", "retrieved"], path, kb_tag=tag, only=ABLATION_MODELS))

In [ ]:
# Tổng hợp: bảng pass@1, so sánh cặp, KB giúp model mạnh hay yếu nhiều hơn.
from IPython.display import Markdown, Image, display
run_live([sys.executable, "tools/summarize_matrix.py", "--results_dir", RESULTS_DIR, "--roster", "kaggle/roster.json"])
summary_md = pathlib.Path(RESULTS_DIR) / "summary" / "matrix_summary.md"
if summary_md.exists():
    display(Markdown(summary_md.read_text(encoding="utf-8")))
png = pathlib.Path(RESULTS_DIR) / "summary" / "matrix_delta.png"
if png.exists():
    display(Image(str(png)))

In [ ]:
# Đóng gói kết quả để tải về (tab Output của notebook). Chép vào Data/results/kaggle/ của project trên máy.
OLLAMA.terminate()
archive = shutil.make_archive("/kaggle/working/cs2311_results", "zip", RESULTS_DIR)
print(archive, round(os.path.getsize(archive) / 1e6, 1), "MB")
# Log từng lệnh đã nằm trong zip: gom thành một file để output ít file (Kaggle giới hạn số file của output)
logs = pathlib.Path(RESULTS_DIR) / "logs"
if logs.exists():
    shutil.make_archive(str(pathlib.Path(RESULTS_DIR) / "logs"), "zip", logs)
    shutil.rmtree(logs)
if LLM_CACHE.exists() and any(LLM_CACHE.iterdir()):
    shutil.make_archive("/kaggle/working/llm_cache", "zip", LLM_CACHE)
n_files = sum(1 for p in pathlib.Path("/kaggle/working").rglob("*") if p.is_file())
size_gb = sum(p.stat().st_size for p in pathlib.Path("/kaggle/working").rglob("*") if p.is_file()) / 1e9
print(f"/kaggle/working: {n_files} file, {size_gb:.2f} GB (Kaggle lưu tối đa 20 GB)")